# Prédiction du SoH des batteries avec un LSTM

## But du travail

Dans ce notebook, on cherche à prédire le **SoH** (*State of Health*) d'une batterie à partir de mesures prises pendant la décharge :

- la tension ;
- le courant ;
- la température ;
- le SoC ;
- le numéro du cycle.

Le problème est un problème de **régression** sur **séries temporelles multivariées**.  
Le choix d'un **LSTM** est donc logique, car ce type de réseau tient compte de l'ordre des mesures.

## Démarche suivie

1. charger et vérifier la base ;
2. observer rapidement la structure des cycles ;
3. créer des fenêtres glissantes ;
4. faire une vraie séparation train / validation / test ;
5. normaliser sans fuite de données ;
6. entraîner un modèle LSTM ;
7. évaluer les résultats ;
8. répondre simplement aux questions du sujet.

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True

print("TensorFlow :", tf.__version__)

## 2. Chargement de la base

In [ ]:
# Le notebook essaie plusieurs noms de fichier pour éviter un blocage inutile.
possible_files = [
    "battery_health_dataset.csv",
    "76d3946d-1198-4ef0-b7f5-3d21870da910.csv"
]

csv_path = None
for name in possible_files:
    if Path(name).exists():
        csv_path = Path(name)
        break

if csv_path is None:
    raise FileNotFoundError(
        "CSV introuvable. Place le fichier dans le même dossier que le notebook "
        "ou renomme-le en 'battery_health_dataset.csv'."
    )

df = pd.read_csv(csv_path)
print("Fichier chargé :", csv_path.name)
df.head()

## 3. Vérification rapide de la structure

Avant de construire le modèle, il faut d'abord s'assurer que la base est cohérente :

- combien il y a de lignes et de colonnes ;
- combien il y a de batteries ;
- combien il y a de cycles au total ;
- si chaque cycle contient bien le même nombre de mesures ;
- s'il manque des valeurs.

In [ ]:
print("Dimensions :", df.shape)
print("Colonnes :", df.columns.tolist())
print("\nValeurs manquantes :")
print(df.isna().sum())

cycle_summary = (
    df.groupby(["battery_id", "cycle_number"])
      .agg(
          n_bins=("SoH", "size"),
          soh_nunique=("SoH", "nunique"),
          soh=("SoH", "first")
      )
      .reset_index()
)

print("\nNombre de batteries :", df["battery_id"].nunique())
print("Nombre total de cycles :", cycle_summary.shape[0])
print("Nombre de bins par cycle :", cycle_summary["n_bins"].unique())
print("Le SoH est constant dans chaque cycle :", bool((cycle_summary["soh_nunique"] == 1).all()))

## 4. Petite exploration

Ici, l'idée n'est pas de faire une exploration très longue.  
On veut juste voir :

- comment le SoH évolue au fil des cycles pour quelques batteries ;
- à quoi ressemble un cycle de décharge sur les variables principales.

In [ ]:
soh_cycle = df.groupby(["battery_id", "cycle_number"])["SoH"].first().reset_index()

batteries_a_montrer = sorted(df["battery_id"].unique())[:4]

plt.figure(figsize=(12, 5))
for bat in batteries_a_montrer:
    d = soh_cycle[soh_cycle["battery_id"] == bat]
    plt.plot(d["cycle_number"], d["SoH"], label=bat, linewidth=2)

plt.axhline(80, linestyle="--", linewidth=1.5, label="Seuil 80")
plt.title("Évolution du SoH sur quelques batteries")
plt.xlabel("Cycle")
plt.ylabel("SoH")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
battery_example = sorted(df["battery_id"].unique())[0]
cycle_example = 1

cycle_data = df[(df["battery_id"] == battery_example) & (df["cycle_number"] == cycle_example)].copy()

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

cols = ["Voltage_measured", "Current_measured", "Temperature_measured", "SoC"]
titles = ["Tension", "Courant", "Température", "SoC"]

for ax, col, title in zip(axes.ravel(), cols, titles):
    ax.plot(cycle_data[col].values, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Bin")

plt.suptitle(f"Exemple de cycle - {battery_example} / cycle {cycle_example}", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Choix des variables et séparation des données

On utilise les variables suivantes comme entrées :

- `Voltage_measured`
- `Current_measured`
- `Temperature_measured`
- `SoC`
- `cycle_number`

La cible à prédire est :

- `SoH`

### Point important

Dans le notebook initial, la normalisation était faite **avant** la séparation train / test.  
Ce n'est pas une bonne pratique, car cela fait fuiter de l'information du test vers le train.

Ici, on corrige cela :

1. on sépare d'abord les batteries ;
2. on crée les fenêtres ;
3. on ajuste les scalers **uniquement sur le train**.

In [ ]:
features = ["Voltage_measured", "Current_measured", "Temperature_measured", "SoC", "cycle_number"]
target = "SoH"

all_batteries = sorted(df["battery_id"].unique())

# Répartition simple et fixe pour que le travail reste reproductible
test_batteries = all_batteries[-4:]        # 4 batteries pour le test
val_batteries = all_batteries[-8:-4]       # 4 batteries pour la validation
train_batteries = all_batteries[:-8]       # le reste pour l'entraînement

print("Train :", train_batteries)
print("Validation :", val_batteries)
print("Test :", test_batteries)

## 6. Création des fenêtres glissantes

In [ ]:
def create_windows(dataframe, feature_cols, target_col, battery_list, window_size=10):
    subset = dataframe[dataframe["battery_id"].isin(battery_list)].copy()
    subset = subset.sort_values(["battery_id", "cycle_number", "SoC"], ascending=[True, True, False])

    X_list, y_list, battery_ids = [], [], []

    for (bat_id, cycle_num), group in subset.groupby(["battery_id", "cycle_number"]):
        values = group[feature_cols].to_numpy(dtype=np.float32)
        soh = float(group[target_col].iloc[0])

        for i in range(len(values) - window_size + 1):
            X_list.append(values[i:i+window_size])
            y_list.append(soh)
            battery_ids.append(bat_id)

    return np.array(X_list), np.array(y_list, dtype=np.float32), np.array(battery_ids)

WINDOW_SIZE = 10

X_train_raw, y_train_raw, bat_train = create_windows(df, features, target, train_batteries, WINDOW_SIZE)
X_val_raw, y_val_raw, bat_val = create_windows(df, features, target, val_batteries, WINDOW_SIZE)
X_test_raw, y_test_raw, bat_test = create_windows(df, features, target, test_batteries, WINDOW_SIZE)

print("Train :", X_train_raw.shape, y_train_raw.shape)
print("Validation :", X_val_raw.shape, y_val_raw.shape)
print("Test :", X_test_raw.shape, y_test_raw.shape)

Avec une fenêtre de 10 bins sur des cycles de 20 bins, chaque cycle produit 11 exemples.  
Cela permet d'augmenter la quantité de données sans modifier la cible du cycle.

## 7. Normalisation correcte

In [ ]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

# On ajuste les scalers uniquement sur le train
scaler_X.fit(X_train_raw.reshape(-1, X_train_raw.shape[-1]))
scaler_y.fit(y_train_raw.reshape(-1, 1))

def transform_X(X, scaler):
    shape = X.shape
    X_flat = X.reshape(-1, shape[-1])
    X_scaled = scaler.transform(X_flat)
    return X_scaled.reshape(shape)

X_train = transform_X(X_train_raw, scaler_X)
X_val = transform_X(X_val_raw, scaler_X)
X_test = transform_X(X_test_raw, scaler_X)

y_train = scaler_y.transform(y_train_raw.reshape(-1, 1)).flatten()
y_val = scaler_y.transform(y_val_raw.reshape(-1, 1)).flatten()
y_test = scaler_y.transform(y_test_raw.reshape(-1, 1)).flatten()

print("Normalisation terminée.")

## 8. Construction du modèle LSTM

Le modèle reste volontairement simple :

- une première couche LSTM pour lire la séquence ;
- une deuxième couche LSTM pour condenser l'information ;
- un dropout pour limiter le surapprentissage ;
- une sortie Dense pour produire une valeur de SoH.

In [ ]:
tf.random.set_seed(42)

model = keras.Sequential([
    layers.Input(shape=(WINDOW_SIZE, len(features))),
    layers.LSTM(64, return_sequences=True),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(1)
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model.summary()

## 9. Entraînement

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

print("Nombre d'époques exécutées :", len(history.history["loss"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history["loss"], label="Train", linewidth=2)
axes[0].plot(history.history["val_loss"], label="Validation", linewidth=2)
axes[0].set_title("Évolution de la loss")
axes[0].set_xlabel("Époque")
axes[0].legend()

axes[1].plot(history.history["mae"], label="Train", linewidth=2)
axes[1].plot(history.history["val_mae"], label="Validation", linewidth=2)
axes[1].set_title("Évolution de la MAE")
axes[1].set_xlabel("Époque")
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Évaluation finale

In [ ]:
y_pred_scaled = model.predict(X_test).flatten()

y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_true = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

baseline_pred = np.repeat(y_train_raw.mean(), len(y_true))
baseline_mae = mean_absolute_error(y_true, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_true, baseline_pred))
baseline_r2 = r2_score(y_true, baseline_pred)

print("=" * 40)
print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R²   : {r2:.4f}")
print("=" * 40)

print("\nBaseline simple (moyenne du train)")
print(f"MAE  : {baseline_mae:.3f}")
print(f"RMSE : {baseline_rmse:.3f}")
print(f"R²   : {baseline_r2:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_true, y_pred, alpha=0.35, s=12)
lims = [min(y_true.min(), y_pred.min()) - 2, max(y_true.max(), y_pred.max()) + 2]
axes[0].plot(lims, lims, "--", linewidth=2)
axes[0].set_title("SoH réel vs SoH prédit")
axes[0].set_xlabel("SoH réel")
axes[0].set_ylabel("SoH prédit")

errors = y_pred - y_true
axes[1].hist(errors, bins=40)
axes[1].axvline(0, linestyle="--", linewidth=2)
axes[1].set_title("Distribution des erreurs")
axes[1].set_xlabel("Erreur")
axes[1].set_ylabel("Fréquence")

plt.tight_layout()
plt.show()

In [ ]:
battery_to_plot = test_batteries[0]
mask_battery = (bat_test == battery_to_plot)

plt.figure(figsize=(12, 5))
plt.plot(y_true[mask_battery], label="SoH réel", linewidth=2)
plt.plot(y_pred[mask_battery], label="SoH prédit", linewidth=2, linestyle="--")
plt.title(f"Suivi du SoH sur la batterie {battery_to_plot}")
plt.xlabel("Fenêtres")
plt.ylabel("SoH")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Réponses aux questions du sujet

### 1) Pourquoi le SoC est-il une variable importante pour estimer le SoH ?

Le SoC indique le niveau de charge de la batterie au moment où les autres mesures sont prises.  
Pris seul, il n'explique pas tout. En revanche, combiné avec la tension, le courant et la température, il aide à mieux situer le comportement de la batterie pendant la décharge.

### 2) Quel est l'intérêt de découper un cycle en plusieurs fenêtres ?

Cela permet d'obtenir plusieurs exemples à partir d'un seul cycle.  
On augmente donc la taille de la base d'apprentissage et on capte des comportements locaux du signal.

### 3) Que se passe-t-il si la fenêtre est trop courte ou trop longue ?

- si elle est trop courte, le modèle manque de contexte ;
- si elle est trop longue, il y a plus de redondance et moins d'exemples distincts.

Il faut donc trouver un compromis entre quantité d'information et quantité d'exemples.

### 4) Quels sont les risques si la séparation train / test est mal faite ?

Le principal risque est une évaluation trop optimiste.  
Si des fenêtres très proches se retrouvent à la fois dans le train et dans le test, le modèle semble meilleur qu'il ne l'est vraiment.

### 5) Dans quels cas industriels ce type de modèle peut-il être utile ?

Ce type de modèle peut être utile dans :

- les véhicules électriques ;
- les systèmes de stockage d'énergie ;
- les objets connectés ;
- le suivi de batteries reconditionnées.

Dans tous ces cas, l'idée est d'anticiper l'usure et d'aider à la maintenance.

## 12. Conclusion

Le notebook de départ avait une bonne idée générale, mais il y avait deux points à corriger :

- le nom du fichier CSV n'était pas garanti ;
- la normalisation était faite avant la séparation des données.

Dans cette version, la démarche est plus propre :

- séparation d'abord ;
- normalisation ensuite ;
- validation explicite ;
- explications plus simples et plus naturelles.

Le travail reste clair, lisible et plus crédible pour un rendu étudiant.